# ECG Morphology Feature Extraction (Tier 2 Clinical Features)

Extracts waveform-derived features from 12-lead ECG NPZ files using neurokit2:

| Feature | Clinical meaning | Lead |
|---|---|---|
| `p_amp_II_uv` | P-wave amplitude — P mitrale marker (LA enlargement) | II |
| `t_amp_V5_uv` | T-wave amplitude — LV strain (negative = inversion) | V5 |
| `t_amp_V4_uv` | T-wave amplitude lateral | V4 |
| `r_amp_I_uv` / `s_amp_I_uv` | QRS components for axis | I |
| `r_amp_aVF_uv` / `s_amp_aVF_uv` | QRS components for axis | aVF |
| `qrs_axis_deg` | QRS electrical axis — right axis (MS/RV load), left axis (MR/LVH) | derived |
| `p_amp_V1_uv` | P-wave amplitude in V1 — terminal negative = PTF-V1 proxy | V1 |
| `p_frac_II` | Fraction beats with detected P in II — low = AF | II |

Output: `ecg_morphology_features.csv` — merges with existing `ecg_intervals.csv`.

In [ ]:
import os, warnings
os.environ['TORCHDYNAMO_DISABLE'] = '1'
warnings.filterwarnings('ignore')

import json, time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import numpy as np
import pandas as pd
import neurokit2 as nk
from tqdm.auto import tqdm

PROJECT_ROOT  = Path('/kaggle/input/datasets/pascalhafidzfajri/mlcv-mitral')
NPZ_DIR       = PROJECT_ROOT / 'mitral_images' / 'mitral_images'
SPLIT_CSV     = PROJECT_ROOT / 'data_split_3level_exp10.csv'
INTERVALS_CSV = PROJECT_ROOT / 'ecg_intervals.csv'
OUT_CSV       = Path('.') / 'ecg_morphology_features.csv'
CHECKPOINT    = Path('.') / 'morph_checkpoint.csv'  # incremental save

FS            = 500   # sampling rate Hz
N_WORKERS     = 4     # parallel workers
SAVE_EVERY    = 500   # save checkpoint every N samples

# Lead indices in NPZ (order: I II III aVR aVL aVF V1 V2 V3 V4 V5 V6)
LEADS = ['I','II','III','aVR','aVL','aVF','V1','V2','V3','V4','V5','V6']
LEAD_IDX = {l: i for i, l in enumerate(LEADS)}
EXTRACT_LEADS = ['I', 'II', 'aVF', 'V1', 'V4', 'V5']  # only process these 6

print('neurokit2:', nk.__version__)
print('NPZ_DIR exists:', NPZ_DIR.exists())
print('INTERVALS_CSV exists:', INTERVALS_CSV.exists())


## 1. Load sample list

In [ ]:
split = pd.read_csv(SPLIT_CSV)
split['stem'] = split['filename'].map(lambda x: Path(str(x)).stem)
split['npz_path'] = split['stem'].map(lambda s: str(NPZ_DIR / f'{s}.npz'))
split['npz_exists'] = split['npz_path'].map(lambda p: Path(p).exists())
avail = split[split['npz_exists']].copy().reset_index(drop=True)
print(f'Available: {len(avail)} / {len(split)} samples')

# Skip already processed if checkpoint exists
done_stems = set()
if CHECKPOINT.exists():
    done_df = pd.read_csv(CHECKPOINT)
    done_stems = set(done_df['stem'].astype(str))
    print(f'Checkpoint: {len(done_stems)} already done')

todo = avail[~avail['stem'].astype(str).isin(done_stems)].copy().reset_index(drop=True)
print(f'To process: {len(todo)} samples')


## 2. Extraction functions

In [ ]:
def extract_lead_features(clean_sig):
    """
    Extract P/R/S/T amplitudes from one cleaned ECG lead.
    Baseline = median of PR-segment windows before each R-peak.
    Returns dict with median amplitudes and detection fraction.
    """
    out = {'p_amp': np.nan, 'r_amp': np.nan, 's_amp': np.nan,
           't_amp': np.nan, 'n_beats': 0, 'p_frac': np.nan}
    if np.std(clean_sig) < 1e-6:
        return out
    try:
        _, rpeaks_d = nk.ecg_peaks(clean_sig, sampling_rate=FS, method='pantompkins1985')
        rp = rpeaks_d['ECG_R_Peaks']
        if len(rp) < 2:
            return out
        _, waves = nk.ecg_delineate(clean_sig, rpeaks_d, sampling_rate=FS,
                                     method='dwt', show=False)
        # PR-segment baseline (60 ms window ending 10 ms before R)
        pr_start = int(0.07 * FS); pr_end = int(0.01 * FS)
        baselines = []
        for r in rp:
            s = max(0, r - pr_start); e = max(1, r - pr_end)
            baselines.append(np.median(clean_sig[s:e]))
        baseline = np.median(baselines)

        def med_amp(locs):
            locs = np.array(locs, dtype=float)
            valid = locs[~np.isnan(locs)].astype(int)
            valid = valid[(valid >= 0) & (valid < len(clean_sig))]
            return (float(np.median(clean_sig[valid] - baseline))
                    if len(valid) >= 2 else np.nan), len(valid)

        out['r_amp'], _   = med_amp(rp)
        out['s_amp'], _   = med_amp(waves['ECG_S_Peaks'])
        out['t_amp'], _   = med_amp(waves['ECG_T_Peaks'])
        out['p_amp'], np_det = med_amp(waves['ECG_P_Peaks'])
        out['n_beats']  = len(rp)
        out['p_frac']   = np_det / len(rp) if len(rp) > 0 else np.nan
    except Exception:
        pass
    return out


def process_sample(row):
    """
    Load one NPZ and extract features from target leads.
    Returns flat dict keyed by lead-specific feature names.
    """
    stem = str(row['stem'])
    result = {'stem': stem}
    try:
        with np.load(str(row['npz_path']), allow_pickle=False) as d:
            sig = d['signal_matrix'].astype(np.float32)  # (T, 12)
            mask = d['overlap_mask'].astype(bool)         # (T, 12)
        sig[mask] = 0.0

        for lead in EXTRACT_LEADS:
            li = LEAD_IDX[lead]
            raw = sig[:, li].astype(np.float64)
            clean = nk.ecg_clean(raw, sampling_rate=FS)
            feats = extract_lead_features(clean)
            sfx = lead.lower().replace('avf', 'avf').replace('avr', 'avr').replace('avl', 'avl')
            for k, v in feats.items():
                result[f'{k}_{sfx}'] = v

        # Derived: QRS electrical axis
        net_I   = (result.get('r_amp_i',   np.nan) or np.nan) + \
                  (result.get('s_amp_i',   np.nan) or np.nan)
        net_aVF = (result.get('r_amp_avf', np.nan) or np.nan) + \
                  (result.get('s_amp_avf', np.nan) or np.nan)
        if np.isfinite(net_I) and np.isfinite(net_aVF) and (net_I != 0 or net_aVF != 0):
            result['qrs_axis_deg'] = float(np.degrees(np.arctan2(net_aVF, net_I)))
        else:
            result['qrs_axis_deg'] = np.nan

    except Exception as e:
        result['error'] = str(e)
    return result


# Quick smoke test
if len(todo) > 0:
    test_row = avail.iloc[0]
    t0 = time.time()
    r = process_sample(test_row)
    print(f'Smoke test ({time.time()-t0:.2f}s):')
    for k, v in r.items():
        if k not in ('stem', 'error'):
            print(f'  {k}: {v}')


## 3. Run extraction (parallelized)

In [ ]:
all_results = list(done_df.to_dict('records')) if CHECKPOINT.exists() else []

batch_buffer = []
t_start = time.time()

with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
    futures = {ex.submit(process_sample, row): i
               for i, row in todo.iterrows()}
    for future in tqdm(as_completed(futures), total=len(futures),
                       desc='extracting ECG morphology'):
        res = future.result()
        batch_buffer.append(res)
        all_results.append(res)

        if len(batch_buffer) >= SAVE_EVERY:
            pd.DataFrame(all_results).to_csv(CHECKPOINT, index=False)
            batch_buffer = []

# Final save
morph_df = pd.DataFrame(all_results)
morph_df.to_csv(CHECKPOINT, index=False)

elapsed = time.time() - t_start
print(f'Done: {len(morph_df)} samples in {elapsed/60:.1f} min')
print(f'Columns: {list(morph_df.columns)}')
print(morph_df.describe().round(1).to_string())


## 4. Merge with existing intervals and save final CSV

In [ ]:
# Merge with existing ecg_intervals.csv
intervals_df = pd.read_csv(INTERVALS_CSV)
merged = intervals_df.merge(morph_df, on='stem', how='left')
print(f'Merged: {len(merged)} rows, {merged.columns.tolist()}')

# Rename for clarity (amplitude in µV)
rename = {
    'p_amp_ii':  'p_amp_II_uv',
    'r_amp_i':   'r_amp_I_uv',
    's_amp_i':   's_amp_I_uv',
    'r_amp_avf': 'r_amp_aVF_uv',
    's_amp_avf': 's_amp_aVF_uv',
    'p_amp_v1':  'p_amp_V1_uv',
    't_amp_v4':  't_amp_V4_uv',
    't_amp_v5':  't_amp_V5_uv',
    'p_frac_ii': 'p_frac_II',
    'n_beats_ii':'n_beats_II',
}
merged.rename(columns={k: v for k, v in rename.items() if k in merged.columns}, inplace=True)

merged.to_csv(OUT_CSV, index=False)
print(f'Saved: {OUT_CSV}  ({len(merged)} rows, {len(merged.columns)} cols)')
print('\nMissing rates:')
new_cols = [c for c in merged.columns if c not in intervals_df.columns]
print(merged[new_cols].isna().mean().round(3).to_string())


## 5. Validation — per-class statistics

In [ ]:
from scipy import stats as sp_stats
from pathlib import Path

# Add class labels
split2 = pd.read_csv(SPLIT_CSV)
split2['stem'] = split2['filename'].map(lambda x: Path(str(x)).stem)
val_df = merged.merge(split2[['stem','flat_label','isHeldOut']], on='stem', how='left')
label_map = {'Kontrol':0,'MR':1,'MS':2}
val_df['cls'] = val_df['flat_label'].map(label_map)
tv = val_df[val_df['isHeldOut']==0].copy()

new_cols = ['p_amp_II_uv','t_amp_V5_uv','t_amp_V4_uv','qrs_axis_deg',
            'p_amp_V1_uv','p_frac_II']
new_cols = [c for c in new_cols if c in tv.columns]

print(f"{'Feature':16s}  {'Ctrl':>10}  {'MR':>10}  {'MS':>10}  {'MR-Ctrl p':>10}  {'MS-Ctrl p':>10}  {'MR-MS p':>9}")
print('-'*85)
for col in new_cols:
    g = [tv[tv.cls==c][col].dropna().values for c in [0,1,2]]
    try:
        _, p1 = sp_stats.mannwhitneyu(g[1], g[0], alternative='two-sided')
        _, p2 = sp_stats.mannwhitneyu(g[2], g[0], alternative='two-sided')
        _, p3 = sp_stats.mannwhitneyu(g[1], g[2], alternative='two-sided')
    except Exception:
        p1=p2=p3=float('nan')
    print(f"{col:16s}  {np.nanmedian(g[0]):>10.1f}  {np.nanmedian(g[1]):>10.1f}  "
          f"{np.nanmedian(g[2]):>10.1f}  {p1:>10.4f}  {p2:>10.4f}  {p3:>9.4f}")

print('\n=== Expected clinical patterns ===')
print('p_amp_II:  MS < Ctrl < MR  (P mitrale: tall in MS with sinus, absent in AF-MS)')
print('t_amp_V5:  MR < Ctrl  (LV strain → T-wave flattening/inversion)')
print('qrs_axis:  MS > Ctrl > MR  (right axis in MS, left axis in MR with LVH)')
print('p_frac_II: MS < Ctrl ~ MR  (AF common in MS → fewer detectable P-waves)')


## Notes

**Output file:** `ecg_morphology_features.csv`  
Download and place alongside `ecg_intervals.csv` — update notebooks to load both.

**Recommended extended INTERVAL_COLS** (12 features total):
```python
INTERVAL_COLS = [
    # Original 6 (from ecg_intervals.csv)
    'rr_ms', 'rr_std_ms', 'p_dur_ms', 'qrs_dur_ms', 'qtc_ms', 'sokolow',
    # From ecg_intervals.csv, previously unused
    'sv1_amp', 'rv5_amp',
    # New waveform morphology (from ecg_morphology_features.csv)
    'p_amp_II_uv', 't_amp_V5_uv', 'qrs_axis_deg', 'p_frac_II',
]
```

**Derived interaction features** (can be computed in dataset class):
- `rv5_sv1_ratio = rv5_amp / sv1_amp` — QRS axis proxy, strongest MS discriminator (d=0.69)
- `p_absent = (p_frac_II < 0.3).astype(float)` — binary AF flag
